# Module 5: Build Your Own MCP Server

In the previous module you used the Neo4j MCP server. In this module you will build your own MCP server using `FastMCP` and the reusable Python code in `src/workshop`.

By the end of this module you will have a custom MCP server that:

- reuses the Neo4j connection and query helpers in the repo
- exposes domain-specific tools instead of generic graph access only
- can be started from VS Code and used from chat

The main idea is simple: keep database access and Cypher in normal Python functions, and keep the MCP layer very thin.

## Reuse the code that already exists

The repo already has most of what you need:

- `../src/workshop/db.py` loads `.env`, connects to Neo4j, and runs read queries against the configured database.
- `../src/workshop/queries.py` contains reusable business queries such as `people_by_skill` and `similar_people`.
- `../src/workshop/mcp_server.py` shows how to turn those helpers into FastMCP tools.


Your task in this module is not to start from scratch. It is to build a more specific MCP server for this HR graph by reusing those pieces.

In [ ]:
from pathlib import Path

print(Path('../src/workshop/mcp_server.py').read_text())

## Design approach

An MCP server should expose tools that match real user questions.

For this graph, that usually means tools like:

- find people by skill
- find people connected to a domain
- find similar candidate profiles using the vector index
- run a restricted read-only Cypher query when you need flexibility

A useful rule of thumb is:

- keep Cypher close to the data layer
- keep tool names friendly and task-oriented
- keep the MCP server read-only unless you have a strong reason not to

In [ ]:
%%writefile ../src/workshop/custom_mcp_server.py
from __future__ import annotations

from typing import Any

from fastmcp import FastMCP

from workshop.db import run_read_query
from workshop.queries import people_by_skill, similar_people


mcp = FastMCP('from-zero-to-hero-custom')


@mcp.tool()
def ping() -> dict[str, str]:
    """Simple health check for the custom MCP server."""
    return {'status': 'ok'}


@mcp.tool()
def find_people_by_skill(skill_name: str, limit: int = 10) -> list[dict[str, Any]]:
    """Return people connected to the given skill."""
    return people_by_skill(skill_name=skill_name, limit=limit)


@mcp.tool()
def find_people_by_domain(domain_name: str, limit: int = 10) -> list[dict[str, Any]]:
    """Return people who have worked on things in the given domain."""
    query = """
    MATCH (person:Person)-[:BUILT|LED|MANAGED|SHIPPED|PUBLISHED|WON|OPTIMIZED]->(:Thing)-[:IN]->(:Domain {name: $domain_name})
    RETURN DISTINCT person.id AS person_id, coalesce(person.name, person.id) AS display_name
    ORDER BY display_name
    LIMIT $limit
    """
    return run_read_query(query, {'domain_name': domain_name, 'limit': limit})


@mcp.tool()
def semantic_people_search(person_id: str, limit: int = 5) -> list[dict[str, Any]]:
    """Return people who are semantically similar to the given person."""
    return similar_people(person_id=person_id, limit=limit)


def main() -> None:
    mcp.run()


if __name__ == '__main__':
    main()

## Test the query logic before you start the server

Before exposing a function as an MCP tool, test the underlying query in Python. This keeps your debugging surface small.

In [ ]:
from workshop.db import run_read_query

query = """
MATCH (person:Person)-[:BUILT|LED|MANAGED|SHIPPED|PUBLISHED|WON|OPTIMIZED]->(:Thing)-[:IN]->(:Domain {name: $domain_name})
RETURN DISTINCT person.id AS person_id, coalesce(person.name, person.id) AS display_name
ORDER BY display_name
LIMIT $limit
"""

run_read_query(query, {'domain_name': 'AI', 'limit': 5})

## Add the custom server to VS Code

Add another server entry below the existing Neo4j MCP to `.vscode/mcp.json`. Like this:

```json
{
  "servers": {
    "neo4j": {
      "type": "stdio",
      "command": "neo4j-mcp",
      "env": {
        "NEO4J_URI": your_neo4j_uri,
        "NEO4J_USERNAME": your_neo4j_username,
        "NEO4J_PASSWORD": your_neo4j_password,
        "NEO4J_DATABASE": your_neo4j_database,
        "NEO4J_TELEMETRY": "false"
      }
    },
    "workshop-custom": {
      "type": "stdio",
      "command": "/workspaces/from_zero_to_hero/.venv/bin/python",
      "args": ["-m", "workshop.custom_mcp_server"]
    }
  }
}
```

Start the `workshop-custom` server from the MCP view in VS Code. Once it is running, ask chat to call tools like `find_people_by_skill` or `find_people_by_domain`.

## Challenge

Extend `custom_mcp_server.py` with one more tool of your own. A good option is a tool that returns a compact profile for a person, including their skills and the domains they have worked in.

When you add it, test the query in Python first, then expose it with `@mcp.tool()`, then try it from chat.